# GLM-4.7-Flash STEM REAP Pruning v2

**REAP** (Router-weighted Expert Activation Pruning) for MoE models.

**Pipeline:**
- Base: zai-org/GLM-4.7-Flash (64 experts)
- Target: 42 experts (33% pruning)
- Calibration: Siesher/mits-calibration-dataset
- Output: GGUF

**Requirements:** Colab Pro+ with A100 40GB

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {gpu_mem:.1f} GB")
else:
    raise RuntimeError("No CUDA GPU!")

In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu tf-keras -q 2>/dev/null || true
!pip install -q --upgrade pip
!pip install -q datasets huggingface_hub accelerate sentencepiece tqdm safetensors
!pip install -q git+https://github.com/huggingface/transformers.git

import transformers
print(f"[OK] transformers: {transformers.__version__}")

In [ ]:
!mkdir -p /content/models /content/outputs /content/gguf /content/offload
from huggingface_hub import login
login()

## 2. Download Model

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoConfig
import os

MODEL_ID = "zai-org/GLM-4.7-Flash"
MODEL_PATH = "/content/models/glm-4.7-flash"

if not os.path.exists(f"{MODEL_PATH}/config.json"):
    print(f"Downloading {MODEL_ID}...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_PATH,
        ignore_patterns=["*.gguf", "*.md", "*.txt"]
    )

config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f"\n[OK] Model: {config.model_type}")
print(f"Experts: {config.n_routed_experts}")
print(f"Layers: {config.num_hidden_layers}")
print(f"Active per token: {config.num_experts_per_tok}")

## 3. Inspect Expert Format

In [ ]:
# Check expert weight format in original model
from safetensors.torch import load_file
import glob
import os

MODEL_PATH = "/content/models/glm-4.7-flash"

files = sorted(glob.glob(os.path.join(MODEL_PATH, "*.safetensors")))
print(f"Found {len(files)} safetensors files\n")

# Find expert keys in layer 1
expert_keys = []
for f in files:
    weights = load_file(f)
    for key in weights.keys():
        if 'layers.1.mlp.experts' in key:
            expert_keys.append((key, weights[key].shape))
    del weights
    if expert_keys:
        break

print("Expert keys in layer 1:")
for key, shape in sorted(expert_keys):
    print(f"  {key}: {shape}")

## 4. REAP Pruning

In [ ]:
# ============================================================================
# REAP PRUNING - Fixed for GLM-4.7-Flash stacked expert format
# ============================================================================

import os
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset
from tqdm import tqdm
import gc
import json
from safetensors.torch import save_file, load_file
import glob
import re

# ===== CONFIGURATION =====
OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
MODEL_PATH = "/content/models/glm-4.7-flash"
DATASET_ID = "Siesher/mits-calibration-dataset"
OFFLOAD_FOLDER = "/content/offload"
COMPRESSION_RATIO = 0.33
N_CALIBRATION_SAMPLES = 500

os.makedirs(OFFLOAD_FOLDER, exist_ok=True)


class REAPObserverV2:
    """
    REAP saliency observer for GLM4MoeLite.
    
    GLM-4.7-Flash experts are stored as stacked tensors:
    - experts.gate_up_proj: [n_experts, intermediate*2, hidden]
    - experts.down_proj: [n_experts, hidden, intermediate]
    """
    
    def __init__(self, model, num_layers: int, num_experts: int, num_experts_per_tok: int):
        self.model = model
        self.num_layers = num_layers
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.device = next(model.parameters()).device
        
        # REAP accumulators
        self.reap_sum = torch.zeros(num_layers, num_experts, device=self.device, dtype=torch.float32)
        self.reap_count = torch.zeros(num_layers, num_experts, device=self.device, dtype=torch.float32)
        self.hooks = []
        self.total_tokens = 0
    
    def _compute_expert_output_norm(self, hidden, experts, expert_idx):
        """
        Compute L2 norm of expert output.
        
        GLM4 expert format:
        - gate_up_proj[expert_idx]: [intermediate*2, hidden]
        - down_proj[expert_idx]: [hidden, intermediate]
        """
        # hidden: [batch, hidden_size]
        gate_up_weight = experts.gate_up_proj[expert_idx]  # [intermediate*2, hidden]
        gate_up = F.linear(hidden.float(), gate_up_weight.float())  # [batch, intermediate*2]
        
        mid = gate_up.shape[-1] // 2
        gate = gate_up[..., :mid]
        up = gate_up[..., mid:]
        activated = F.silu(gate) * up  # [batch, intermediate]
        
        down_weight = experts.down_proj[expert_idx]  # [hidden, intermediate]
        output = F.linear(activated, down_weight.float())  # [batch, hidden]
        
        return torch.linalg.norm(output, dim=-1)  # [batch]
    
    def _create_hook(self, layer_idx: int):
        def hook(module, args, output):
            hidden_states = args[0]
            if hidden_states.dim() == 2:
                hidden_states = hidden_states.unsqueeze(0)
            
            batch_size, seq_len, hidden_size = hidden_states.shape
            num_tokens = batch_size * seq_len
            self.total_tokens += num_tokens
            
            # Get routing weights
            router_logits = module.gate(hidden_states)  # [batch, seq, n_experts]
            routing_weights = F.softmax(router_logits, dim=-1, dtype=torch.float32)
            topk_weights, topk_indices = torch.topk(routing_weights, self.num_experts_per_tok, dim=-1)
            
            hidden_flat = hidden_states.view(num_tokens, hidden_size)
            topk_indices_flat = topk_indices.view(num_tokens, self.num_experts_per_tok)
            routing_flat = routing_weights.view(num_tokens, self.num_experts)
            experts = module.experts
            
            with torch.no_grad():
                # Process each unique expert that was selected
                unique_experts = topk_indices_flat.unique()
                for expert_idx in unique_experts.tolist():
                    # Mask of tokens that selected this expert
                    active_mask = (topk_indices_flat == expert_idx).any(dim=-1)
                    if not active_mask.any():
                        continue
                    
                    active_hidden = hidden_flat[active_mask]  # [n_active, hidden]
                    active_weights = routing_flat[active_mask, expert_idx]  # [n_active]
                    
                    try:
                        expert_norms = self._compute_expert_output_norm(active_hidden, experts, expert_idx)
                        # REAP score: router_weight * output_norm
                        reap_contribution = (active_weights * expert_norms).sum()
                        
                        if not torch.isnan(reap_contribution) and not torch.isinf(reap_contribution):
                            self.reap_sum[layer_idx, expert_idx] += reap_contribution
                            self.reap_count[layer_idx, expert_idx] += active_mask.sum().float()
                    except Exception as e:
                        # Skip problematic experts
                        pass
        return hook
    
    def register_hooks(self):
        moe_layers = 0
        for layer_idx in range(self.num_layers):
            layer = self.model.model.layers[layer_idx]
            if hasattr(layer.mlp, 'experts'):
                hook = layer.mlp.register_forward_hook(self._create_hook(layer_idx))
                self.hooks.append(hook)
                moe_layers += 1
        print(f"Registered hooks on {moe_layers} MoE layers")
    
    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
    
    def get_global_saliency(self):
        """Compute global REAP saliency averaged across layers."""
        # Mean saliency per expert per layer
        layer_means = self.reap_sum / self.reap_count.clamp(min=1)
        
        # Find layers with data
        valid_layers = (self.reap_count.sum(dim=1) > 0)
        
        if valid_layers.sum() > 0:
            # Average across valid layers
            global_saliency = layer_means[valid_layers].mean(dim=0)
        else:
            global_saliency = layer_means.mean(dim=0)
        
        return global_saliency
    
    def get_experts_to_keep(self, target_experts: int):
        """Get top-k experts by saliency."""
        saliency = self.get_global_saliency()
        
        # Handle NaN - replace with min valid value
        valid_mask = ~torch.isnan(saliency)
        if valid_mask.sum() == 0:
            print("WARNING: All saliency values are NaN! Using uniform selection.")
            return list(range(target_experts))
        
        min_valid = saliency[valid_mask].min()
        saliency = torch.where(torch.isnan(saliency), min_valid, saliency)
        
        _, top_indices = torch.topk(saliency, target_experts)
        return sorted(top_indices.cpu().tolist())
    
    def print_stats(self):
        """Print calibration statistics."""
        print(f"\nCalibration Stats:")
        print(f"  Total tokens: {self.total_tokens:,}")
        print(f"  Layers with data: {(self.reap_count.sum(dim=1) > 0).sum().item()}")
        print(f"  Experts activated: {(self.reap_count.sum(dim=0) > 0).sum().item()}")
        
        saliency = self.get_global_saliency()
        valid = ~torch.isnan(saliency)
        if valid.sum() > 0:
            print(f"  Saliency range: [{saliency[valid].min():.4f}, {saliency[valid].max():.4f}]")
            print(f"  NaN experts: {(~valid).sum().item()}")
        else:
            print(f"  WARNING: All saliency values are NaN!")


def prune_and_save_stacked_experts(model_path, experts_to_keep, target_experts, output_dir):
    """
    Prune experts stored as stacked tensors.
    
    GLM-4.7-Flash format:
    - model.layers.X.mlp.experts.gate_up_proj: [64, intermediate*2, hidden]
    - model.layers.X.mlp.experts.down_proj: [64, hidden, intermediate]
    """
    experts_to_keep_sorted = sorted(experts_to_keep)
    print(f"Keeping {len(experts_to_keep_sorted)} experts: {experts_to_keep_sorted}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Update config
    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    config.n_routed_experts = target_experts
    config.save_pretrained(output_dir)
    
    # Copy tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tokenizer.save_pretrained(output_dir)
    
    safetensor_files = sorted(glob.glob(os.path.join(model_path, "*.safetensors")))
    print(f"Processing {len(safetensor_files)} weight files...")
    
    pruned_state_dict = {}
    
    for sf_path in tqdm(safetensor_files, desc="Loading & pruning"):
        weights = load_file(sf_path)
        
        for key, value in weights.items():
            if '.mlp.experts.gate_up_proj' in key or '.mlp.experts.down_proj' in key:
                # Stacked expert weights: [n_experts, ...]
                # Select only kept experts
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()
            
            elif '.mlp.gate.weight' in key:
                # Router weights: [n_experts, hidden]
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()
            
            elif '.mlp.gate.e_score_correction_bias' in key:
                # Router bias: [n_experts]
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()
            
            else:
                # Keep as-is
                pruned_state_dict[key] = value
        
        del weights
        gc.collect()
    
    # Save with sharding
    print(f"Saving {len(pruned_state_dict)} tensors...")
    total_size = sum(t.numel() * t.element_size() for t in pruned_state_dict.values())
    max_shard_size = 5 * 1024 * 1024 * 1024
    
    if total_size <= max_shard_size:
        save_file(pruned_state_dict, os.path.join(output_dir, "model.safetensors"))
    else:
        current_shard, current_size, shard_idx = {}, 0, 1
        index = {"weight_map": {}, "metadata": {"total_size": total_size}}
        
        for key, tensor in tqdm(pruned_state_dict.items(), desc="Sharding"):
            tensor_size = tensor.numel() * tensor.element_size()
            if current_size + tensor_size > max_shard_size and current_shard:
                shard_name = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
                save_file(current_shard, os.path.join(output_dir, shard_name))
                shard_idx += 1
                current_shard, current_size = {}, 0
            current_shard[key] = tensor
            current_size += tensor_size
            index["weight_map"][key] = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
        
        if current_shard:
            save_file(current_shard, os.path.join(output_dir, f"model-{shard_idx:05d}-of-XXXXX.safetensors"))
        
        total_shards = shard_idx
        for key in index["weight_map"]:
            index["weight_map"][key] = index["weight_map"][key].replace("XXXXX", f"{total_shards:05d}")
        
        for i in range(1, total_shards + 1):
            old = os.path.join(output_dir, f"model-{i:05d}-of-XXXXX.safetensors")
            new = os.path.join(output_dir, f"model-{i:05d}-of-{total_shards:05d}.safetensors")
            if os.path.exists(old):
                os.rename(old, new)
        
        with open(os.path.join(output_dir, "model.safetensors.index.json"), "w") as f:
            json.dump(index, f, indent=2)
    
    print(f"[OK] Saved to {output_dir}")


def run_reap_pruning():
    """Main REAP pruning pipeline."""
    print("="*60)
    print("GLM-4.7-Flash REAP Pruning")
    print("="*60)
    
    config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
    num_experts = config.n_routed_experts
    num_layers = config.num_hidden_layers
    num_experts_per_tok = config.num_experts_per_tok
    target_experts = int(num_experts * (1 - COMPRESSION_RATIO))
    
    print(f"Experts: {num_experts} -> {target_experts} ({COMPRESSION_RATIO*100:.0f}% reduction)")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print("\nLoading model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        offload_folder=OFFLOAD_FOLDER
    )
    model.eval()
    
    # Verify expert format
    l1 = model.model.layers[1].mlp
    print(f"\nExpert format check:")
    print(f"  gate_up_proj shape: {l1.experts.gate_up_proj.shape}")
    print(f"  down_proj shape: {l1.experts.down_proj.shape}")
    
    dataset = load_dataset(DATASET_ID, split="train")
    print(f"\nDataset: {len(dataset)} samples")
    
    observer = REAPObserverV2(model, num_layers, num_experts, num_experts_per_tok)
    observer.register_hooks()
    
    print(f"\nCalibrating on {N_CALIBRATION_SAMPLES} samples...")
    with torch.no_grad():
        for i in tqdm(range(min(N_CALIBRATION_SAMPLES, len(dataset)))):
            text = dataset[i]['instruction']
            if dataset[i].get('output'):
                text += "\n" + dataset[i]['output'][:500]
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            try:
                model(**{k: v.to(model.device) for k, v in inputs.items()})
            except:
                continue
    
    observer.remove_hooks()
    observer.print_stats()
    
    experts_to_keep = observer.get_experts_to_keep(target_experts)
    saliency = observer.get_global_saliency()
    
    # Free memory
    del model, observer
    gc.collect()
    torch.cuda.empty_cache()
    print("\nGPU memory freed.")
    
    # Prune and save
    print("\nPruning and saving...")
    prune_and_save_stacked_experts(MODEL_PATH, experts_to_keep, target_experts, OUTPUT_DIR)
    
    # Save metadata
    metadata = {
        "base_model": "zai-org/GLM-4.7-Flash",
        "method": "REAP",
        "calibration_dataset": DATASET_ID,
        "calibration_samples": N_CALIBRATION_SAMPLES,
        "original_experts": num_experts,
        "pruned_experts": target_experts,
        "compression_ratio": COMPRESSION_RATIO,
        "experts_kept": experts_to_keep,
        "saliency_scores": saliency.cpu().tolist()
    }
    with open(f"{OUTPUT_DIR}/reap_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print("\n" + "="*60)
    print("REAP Pruning Complete!")
    print(f"Experts kept: {experts_to_keep}")
    print("="*60)
    
    return experts_to_keep

In [ ]:
# Clean and run
!rm -rf /content/outputs/glm-stem-pruned /content/offload

experts_kept = run_reap_pruning()

In [ ]:
# FORCE RESTART KERNEL before verification
print("Restarting kernel to free GPU memory...")
print("After restart, run the VERIFICATION cell below.")

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

---
## 5. Verification (Run AFTER kernel restart)

In [ ]:
# Verify structure without loading full model
from safetensors.torch import load_file
from transformers import AutoConfig
import glob
import os

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"

config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"Config n_routed_experts: {config.n_routed_experts}")

files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")))
print(f"Found {len(files)} safetensors files\n")

# Check expert weights in layer 1
for f in files:
    weights = load_file(f)
    for key in weights.keys():
        if 'layers.1.mlp.experts' in key:
            print(f"{key}: {weights[key].shape}")
        if 'layers.1.mlp.gate.weight' in key:
            print(f"{key}: {weights[key].shape}")
    del weights
    break

print("\n[OK] Structure verified!")

In [ ]:
# Load and test generation
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch
import os

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
OFFLOAD_FOLDER = "/content/offload_verify"
os.makedirs(OFFLOAD_FOLDER, exist_ok=True)

config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"Loading pruned model ({config.n_routed_experts} experts)...")

model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    offload_folder=OFFLOAD_FOLDER
)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

# Quick structure check
l1 = model.model.layers[1].mlp
print(f"\nlayer1 experts: {l1.experts.gate_up_proj.shape[0]}")
print(f"layer1 gate: {l1.gate.weight.shape[0]}")

# Generation test
print("\n" + "="*40)
print("Generation Test")
print("="*40)

prompts = [
    "Solve: 2x + 5 = 13",
    "Write Python code to check if a number is prime:"
]

model.eval()
for p in prompts:
    print(f"\n> {p}")
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(p):].strip()
    print(response[:300])

print("\n[OK] Generation works!")

In [ ]:
# Cleanup before GGUF
try:
    del model
except:
    pass
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("[OK] Memory cleared")

## 6. GGUF Conversion

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp 2>/dev/null || true
!pip install -q gguf numpy
print("[OK] llama.cpp ready")

In [ ]:
import os
OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
GGUF_FP16 = "/content/gguf/glm-stem-42exp-f16.gguf"

%cd /content/llama.cpp
!python convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_FP16}" --outtype f16

if os.path.exists(GGUF_FP16):
    print(f"\n[OK] FP16: {os.path.getsize(GGUF_FP16)/1e9:.2f} GB")

In [ ]:
%cd /content/llama.cpp
!make llama-quantize -j$(nproc) 2>/dev/null || echo "Already compiled"

In [ ]:
import os
GGUF_FP16 = "/content/gguf/glm-stem-42exp-f16.gguf"
GGUF_Q4 = "/content/gguf/glm-stem-42exp-q4km.gguf"
GGUF_Q8 = "/content/gguf/glm-stem-42exp-q8.gguf"

%cd /content/llama.cpp

if os.path.exists(GGUF_FP16):
    !./llama-quantize "{GGUF_FP16}" "{GGUF_Q4}" Q4_K_M
    !./llama-quantize "{GGUF_FP16}" "{GGUF_Q8}" Q8_0
    
    print("\nGGUF files:")
    for f in [GGUF_Q4, GGUF_Q8]:
        if os.path.exists(f):
            print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
    
    os.remove(GGUF_FP16)
    print("\n[OK] Quantization complete!")

## 7. Summary

In [ ]:
import os, json

print("="*60)
print("GLM-4.7-Flash STEM REAP Pruning - COMPLETE")
print("="*60)

try:
    with open("/content/outputs/glm-stem-pruned/reap_metadata.json") as f:
        m = json.load(f)
    print(f"\nMethod: {m['method']}")
    print(f"Experts: {m['original_experts']} -> {m['pruned_experts']}")
    print(f"Compression: {m['compression_ratio']*100:.0f}%")
    print(f"Kept: {m['experts_kept']}")
except Exception as e:
    print(f"Metadata: {e}")

print("\nOutput files:")
for f in ["/content/outputs/glm-stem-pruned", 
          "/content/gguf/glm-stem-42exp-q4km.gguf",
          "/content/gguf/glm-stem-42exp-q8.gguf"]:
    if os.path.exists(f):
        if f.endswith(".gguf"):
            print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
        else:
            print(f"  HF: {f}")

print("\n" + "="*60)